# Person Re-Identification — Experiments

**Dataset**: Market-1501  
**Backbone**: torchreid ResNet50 (Market-1501 pretrained)

| # | Experiment | Idea |
|---|---|---|
| 01 | Dataset EDA | Understand the data |
| 02 | Embedding extraction | ResNet50 → 2048-d L2-normalised vectors |
| 03 | Similarity analysis | Genuine vs impostor distributions |
| 04 | **Baseline**: Cosine similarity ranking | Pure appearance, no threshold |
| 05 | **Exp A**: Cosine + feasibility threshold | Reject low-confidence matches |
| 06 | Threshold sweep | Find optimal operating point |
| 07 | Results summary | Side-by-side comparison |

---

**All logic lives in `src/`** — this notebook only imports and calls.
```
src/
├── dataset.py            ← Market-1501 filename parsing, MarketDataset
├── feature_extractor.py  ← build_torchreid_resnet50, extract_embeddings
├── metrics.py            ← cosine_similarity_matrix, evaluate_ranking, apply_threshold
└── visualization.py      ← all plots
```

## 0. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # project root on path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

from src import (
    # data
    load_splits, print_split_stats,
    # model
    build_torchreid_resnet50, extract_embeddings, verify_normalization,
    # metrics
    cosine_similarity_matrix, apply_threshold,
    evaluate_ranking, print_metrics, sample_pair_similarities,
    # plots
    plot_dataset_eda, plot_similarity_distribution,
    plot_cmc_curves, plot_reid_results,
    plot_threshold_sweep, plot_results_comparison,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Config
**All knobs live here** — never bury magic numbers in cells below.

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────
DATASET_ROOT = "/content/drive/MyDrive/Colab Notebooks/Market-1501-v15.09.15"
OUTPUT_DIR   = "/content/reid_outputs"   # embeddings + figures saved here

# ── Model ───────────────────────────────────────────────────────────────────
BATCH_SIZE   = 512    # safe for A100; reduce to 64 for CPU / small GPU
NUM_CLASSES  = 751    # Market-1501 training identities

# ── Evaluation ──────────────────────────────────────────────────────────────
MAX_RANK     = 10
N_PAIRS      = 2000   # pairs for similarity distribution analysis

# ── Threshold experiment ────────────────────────────────────────────────────
THRESHOLD_RANGE = np.arange(0.0, 0.95, 0.05)   # values to sweep
FIXED_THRESHOLD = 0.30                           # used in Exp A

import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Config ready")

## 2. Load Dataset

In [ ]:
train_df, query_df, gallery_df = load_splits(DATASET_ROOT)

print_split_stats(train_df,   "Train")
print_split_stats(query_df,   "Query")
print_split_stats(gallery_df, "Gallery")

## 3. Dataset EDA

In [ ]:
fig = plot_dataset_eda(train_df, title_suffix="Training Set")
fig.savefig(f"{OUTPUT_DIR}/eda_train.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Feature Extraction
Using **torchreid ResNet50 pretrained on Market-1501** (re-ID weights, not ImageNet-only).

> To switch to plain ImageNet weights, replace `build_torchreid_resnet50()` with `build_torchvision_resnet50()`.

In [ ]:
model = build_torchreid_resnet50(num_classes=NUM_CLASSES, device=device)

In [ ]:
query_emb   = extract_embeddings(query_df,   model, batch_size=BATCH_SIZE, device=device)
gallery_emb = extract_embeddings(gallery_df, model, batch_size=BATCH_SIZE, device=device)

verify_normalization(query_emb)
verify_normalization(gallery_emb)

print(f"\nquery_emb   : {query_emb.shape}")
print(f"gallery_emb : {gallery_emb.shape}")

In [ ]:
# Save embeddings — skip extraction on re-runs
np.save(f"{OUTPUT_DIR}/query_embeddings.npy",   query_emb)
np.save(f"{OUTPUT_DIR}/gallery_embeddings.npy", gallery_emb)
query_df.to_csv(f"{OUTPUT_DIR}/query_metadata.csv",   index=False)
gallery_df.to_csv(f"{OUTPUT_DIR}/gallery_metadata.csv", index=False)
print("Saved embeddings and metadata.")

In [ ]:
# ─── Uncomment to load saved embeddings instead of re-extracting ───
# query_emb   = np.load(f"{OUTPUT_DIR}/query_embeddings.npy")
# gallery_emb = np.load(f"{OUTPUT_DIR}/gallery_embeddings.npy")
# query_df    = pd.read_csv(f"{OUTPUT_DIR}/query_metadata.csv")
# gallery_df  = pd.read_csv(f"{OUTPUT_DIR}/gallery_metadata.csv")

## 5. Similarity Distribution Analysis
Understand the genuine/impostor overlap before running any ranking experiment.
This informs a sensible starting threshold.

In [ ]:
# Sample from training set (has richer identity coverage than query)
# Re-extract train embeddings if needed, or use gallery as proxy.
positive_sims, negative_sims = sample_pair_similarities(
    gallery_df, gallery_emb, n_pairs=N_PAIRS, seed=42
)

print(f"Mean genuine  similarity : {np.mean(positive_sims):.4f}")
print(f"Mean impostor similarity : {np.mean(negative_sims):.4f}")

In [ ]:
fig = plot_similarity_distribution(
    positive_sims, negative_sims,
    title="Gallery: Genuine vs Impostor Cosine Similarity",
    threshold=FIXED_THRESHOLD,
)
fig.savefig(f"{OUTPUT_DIR}/similarity_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Experiment 01 — Baseline: Pure Cosine Similarity

**No threshold.** Every gallery entry is ranked purely by cosine similarity.  
This reproduces the approach at the end of the original notebook (the `@` dot-product matrix).

In [ ]:
query_ids   = query_df["person_id"].to_numpy()
gallery_ids = gallery_df["person_id"].to_numpy()
query_cams  = query_df["camera_id"].to_numpy()
gallery_cams = gallery_df["camera_id"].to_numpy()

sim_matrix = cosine_similarity_matrix(query_emb, gallery_emb)
print(f"Similarity matrix: {sim_matrix.shape}")

In [ ]:
results_baseline = evaluate_ranking(
    sim_matrix, query_ids, gallery_ids, query_cams, gallery_cams,
    max_rank=MAX_RANK,
)
print_metrics(results_baseline, label="Exp 01: Cosine Similarity Baseline")

In [ ]:
# Sanity check: visual retrieval for first query
q_idx = 0
fig = plot_reid_results(
    query_img_path=query_df.iloc[q_idx]["image_path"],
    gallery_df=gallery_df,
    gallery_ids=gallery_ids,
    query_pid=query_ids[q_idx],
    similarities=sim_matrix[q_idx],
    top_k=5,
    title=f"Exp 01 — Query {q_idx} (ID:{query_ids[q_idx]}), Top-5",
)
plt.show()

---
## Experiment 02 — Cosine + Feasibility Threshold

Gallery entries with cosine similarity below `FIXED_THRESHOLD` are treated as  
**infeasible** and excluded from ranking. This prevents the model from ever  
returning a match it has very low confidence in.

The optimal threshold value comes from the sweep in section 6.

In [ ]:
sim_thresholded = apply_threshold(sim_matrix, threshold=FIXED_THRESHOLD)

results_threshold = evaluate_ranking(
    sim_thresholded, query_ids, gallery_ids, query_cams, gallery_cams,
    max_rank=MAX_RANK,
)
print_metrics(results_threshold, label=f"Exp 02: Cosine + Threshold={FIXED_THRESHOLD}")

In [ ]:
# Compare CMC curves side by side
fig = plot_cmc_curves(
    {
        "Baseline (no threshold)": results_baseline,
        f"Threshold={FIXED_THRESHOLD}": results_threshold,
    },
    max_rank=MAX_RANK,
    title="Exp 01 vs Exp 02 — CMC Curves",
)
fig.savefig(f"{OUTPUT_DIR}/cmc_exp01_vs_exp02.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Section 6 — Threshold Sweep

Sweep `THRESHOLD_RANGE` to find the best operating point for Rank-1 and mAP.

In [ ]:
from tqdm.auto import tqdm

rank1_scores, map_scores = [], []

for t in tqdm(THRESHOLD_RANGE, desc="Threshold sweep"):
    sim_t = apply_threshold(sim_matrix, threshold=float(t))
    res_t = evaluate_ranking(
        sim_t, query_ids, gallery_ids, query_cams, gallery_cams,
        max_rank=MAX_RANK,
    )
    rank1_scores.append(res_t["rank1"])
    map_scores.append(res_t["mAP"])

rank1_scores = np.array(rank1_scores)
map_scores   = np.array(map_scores)

best_idx = int(np.argmax(rank1_scores))
best_threshold = THRESHOLD_RANGE[best_idx]
print(f"Best Rank-1 = {rank1_scores[best_idx]:.4f} at threshold = {best_threshold:.2f}")

In [ ]:
fig = plot_threshold_sweep(
    THRESHOLD_RANGE, rank1_scores, map_scores,
    optimal_threshold=best_threshold,
    title="Threshold Sweep — Rank-1 & mAP vs. Cosine Threshold",
)
fig.savefig(f"{OUTPUT_DIR}/threshold_sweep.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Section 7 — Results Summary

In [ ]:
# Re-run with the optimal threshold found above
results_optimal = evaluate_ranking(
    apply_threshold(sim_matrix, best_threshold),
    query_ids, gallery_ids, query_cams, gallery_cams,
    max_rank=MAX_RANK,
)

all_results = {
    "Baseline": results_baseline,
    f"Threshold={FIXED_THRESHOLD}": results_threshold,
    f"Threshold={best_threshold:.2f} (optimal)": results_optimal,
}

# Print table
summary = pd.DataFrame([
    {"Experiment": k, "Rank-1": v["rank1"], "Rank-5": v["rank5"],
     "Rank-10": v["rank10"], "mAP": v["mAP"]}
    for k, v in all_results.items()
]).set_index("Experiment")

print(summary.to_string(float_format="{:.4f}".format))

In [ ]:
fig = plot_results_comparison(
    all_results,
    metrics=["rank1", "rank5", "mAP"],
    title="All Experiments — Performance Comparison",
)
fig.savefig(f"{OUTPUT_DIR}/results_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig = plot_cmc_curves(all_results, max_rank=MAX_RANK, title="All Experiments — CMC Curves")
fig.savefig(f"{OUTPUT_DIR}/cmc_all.png", dpi=120, bbox_inches="tight")
plt.show()

---
## Next Experiments

| # | Method | Expected gain |
|---|---|---|
| Exp 03 | k-reciprocal re-ranking (Zhong et al. 2017) | +5–10% mAP, free |
| Exp 04 | Context features: camera pair + frame diff (your context model) | Depends on calibration |
| Exp 05 | Query expansion (AQE) | +2–5% mAP |
| Exp 06 | OSNet backbone (torchreid) | Stronger baseline |

---

### Note on the context model (from your original notebook)

Your context model (logistic regression on cosine_similarity + log_time_diff + camera_pair)
improved pairwise AUC but **hurt** Rank-1. This is expected: temporal constraints  
penalise genuine matches with large frame gaps. Two cleaner ways to add context:

1. **Use camera pair only** (drop time_diff) — camera transition statistics are stable and don't penalise long-gap matches.
2. **Late fusion** — compute cosine rank + context score separately, fuse with a learned weight α:  
   `final_score = α * cosine_sim + (1-α) * context_score`  
   Sweep α on a held-out val set.